In [1]:
from __future__ import annotations

import sys
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

sys.path.append(str(Path("..").resolve()))

from src.features.transformers import (
    StringCleaner,
    MissingIndicatorAdder,
    StructuralMissingImputer,
    BinaryYesNoEncoder,
    RareCategoryGrouper,
    FeatureEngineer,
)

RANDOM_STATE = 42
TARGET_COL = "Depression"
ID_COL = "id"

train: pd.DataFrame = pd.read_csv("../data/train.csv")
test: pd.DataFrame = pd.read_csv("../data/test.csv")

In [2]:
drop_cols = [ID_COL, "Name"]

X = train.drop(columns= drop_cols + [TARGET_COL])
y = train[TARGET_COL]
X_test = test.drop(columns= drop_cols)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (112560, 17), Val: (28140, 17), Test: (93800, 17)


In [ ]:
pre_pipeline = Pipeline(steps=[
    ("string_cleaner", StringCleaner()),
    ("missing_indicators", MissingIndicatorAdder(min_missing_frac=0.0)),
    ("structural_imputer", StructuralMissingImputer()),
    ("binary_encoder", BinaryYesNoEncoder()),
    ("feature_engineer", FeatureEngineer()),
    ("rare_grouper", RareCategoryGrouper(min_count=30)),
])

X_train_pre = pre_pipeline.fit_transform(X_train)
X_val_pre   = pre_pipeline.transform(X_val)
X_test_pre  = pre_pipeline.transform(X_test)

print("Columns after pre-pipeline:", X_train_pre.shape[1])

Columns after pre-pipeline: 35


In [ ]:
import numpy as np
from numpy.typing import NDArray

numeric_cols: list[str] = X_train_pre.select_dtypes(include=np.number).columns.tolist()
categorical_cols: list[str] = X_train_pre.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

numeric_transformer: Pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_transformer: Pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor: ColumnTransformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

X_train_processed: NDArray = preprocessor.fit_transform(X_train_pre)
X_val_processed: NDArray   = preprocessor.transform(X_val_pre)
X_test_processed: NDArray  = preprocessor.transform(X_test_pre)

print(f"Final shape: {X_train_processed.shape}")

Numeric (28): ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Profession_is_missing', 'Academic Pressure_is_missing', 'Work Pressure_is_missing', 'CGPA_is_missing', 'Study Satisfaction_is_missing', 'Job Satisfaction_is_missing', 'Dietary Habits_is_missing', 'Degree_is_missing', 'Financial Stress_is_missing', 'Total Pressure', 'Max Pressure', 'Total Satisfaction', 'Min Satisfaction', 'Pressure_Satisfaction_Gap', 'High_Working_Hours_Flag', 'High_Financial_Stress_Flag', 'Is_Young_Adult', 'Suicide_Pressure_Interaction']
Categorical (7): ['Gender', 'City', 'Working Professional or Student', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']
Final shape: (112560, 140)


In [7]:
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(pre_pipeline,  "../models/pre_pipeline.joblib")
joblib.dump(preprocessor,  "../models/preprocessor.joblib")

print("Saved pipeline into models/")

Saved pipeline into models/
